In [1]:
from pathlib import Path
from trail_pacer.utils import DATA_PATH
from trail_pacer.gpx_utils import GPXLoader
GPX_FILE = Path(DATA_PATH,"gpx", "TSB2026-LA_BOUCLE_CUGEOISE.gpx")

In [2]:
gpxloader = GPXLoader(GPX_FILE,interval_km=1)
df_gpx = gpxloader.process_gpx()

In [3]:
from trail_pacer.utils import Conversions as conv
conv.min_to_chrono(conv.chrono_to_min("38:51")/10)

'3:53'

In [4]:
from trail_pacer.models import PaceModel

model = PaceModel(pace_10k_min_per_km='3:53', up_sens=0.05, down_sens=0.02, fatigue_rate=0.06)

model.summary(df_gpx)

from datetime import datetime
splits = model.predict_split_times(df_gpx, start_time=datetime(2025, 6, 1, 8, 0))

splits[["distance_km", "elevation_m", "grade_pct", "predicted_pace", "cum_time_str", "arrival_time"]]

Base pace       : 3.88 min/km  (3:53)
up_sens         : 0.05
down_sens       : 0.02
fatigue_rate    : 0.06
Distance        : 25.9 km
Flat estimate   : 1:40:24
Grade-adjusted  : 1:44:53
Grade + fatigue : 2:05:42


,distance_km,elevation_m,grade_pct,predicted_pace,cum_time_str,arrival_time
0,0.000000,223.240000,0.000000,3.883333,0:00:00,2025-06-01 08:00:00.000000000
1,1.000000,189.790097,-3.344990,3.876434,0:03:53,2025-06-01 08:03:52.586011614
2,2.000000,290.503822,10.071373,4.506902,0:08:23,2025-06-01 08:08:23.000129130
3,3.000000,471.747558,18.124374,4.969552,0:13:21,2025-06-01 08:13:21.173250072
4,4.000000,563.381648,9.163409,4.581504,0:17:56,2025-06-01 08:17:56.063476986
5,5.000000,584.935067,2.155342,4.291100,0:22:14,2025-06-01 08:22:13.529502678
6,6.000000,602.136209,1.720114,4.329339,0:26:33,2025-06-01 08:26:33.289845240
7,7.000000,698.103644,9.596743,4.783171,0:31:20,2025-06-01 08:31:20.280075660
8,8.000000,799.745629,10.164198,4.871543,0:36:13,2025-06-01 08:36:12.572671146
9,9.000000,610.455515,-18.929011,4.044753,0:40:15,2025-06-01 08:40:15.257857536


# Comparing Predictions with actual times from Pierre's run

In [5]:
import pandas as pd

df_temps_pierre = pd.read_excel("/Users/sauniere/Desktop/Perso/Codes/Running/trail-pacer/data/Temps_Pierre_Cugeoise.xlsx")
for col in ['Time','GAP /km']:
    df_temps_pierre[col] = pd.to_datetime(df_temps_pierre[col], format="%H:%M:%S")
    df_temps_pierre[col] = df_temps_pierre[col].dt.strftime("%H:%M")

df_temps_pierre["pace"] = df_temps_pierre["Time"].apply(conv.chrono_to_min) 
# Add a column with the cumulated pace up to that point
df_temps_pierre["cum_pace"] = df_temps_pierre["pace"].cumsum()
df_temps_pierre["cum_dist"] = df_temps_pierre["Distance km"].cumsum()
df_temps_pierre.head(2)

,Lap,Distance km,Time,GAP /km,Elev,HR,pace,cum_pace,cum_dist
0,1,1.0,05:13,05:16,-22,146,5.216667,5.216667,1.0
1,2,1.0,07:39,04:48,80,157,7.650000,12.866667,2.0


In [6]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=splits["distance_km"], y=splits["elevation_m"],
        fill="tozeroy", fillcolor="rgba(100,150,255,0.15)",
        line=dict(color="rgba(100,150,255,0.6)", width=1.5),
        name="Elevation (m)",
        hovertemplate="%{y:.0f} m<extra></extra>",
    ),
    secondary_y=False,
)

splits["pace_fmt"]    = splits["predicted_pace"].apply(conv.min_to_chrono)

fig.add_trace(
    go.Scatter(
        x=splits["distance_km"], y=splits["predicted_pace"],
        line=dict(color="tomato", width=2.5),
        name="Predicted Pace",
        customdata=splits["pace_fmt"],
        hovertemplate="%{customdata} min/km<extra></extra>",
    ),
    secondary_y=True,
)
fig.add_trace(
    go.Scatter(
        x=df_temps_pierre["cum_dist"]-1, y=df_temps_pierre["pace"],
        line=dict(color="darkorange", width=2, dash="dot"),
        name="Real Pace",
        # customdata=df_temps_pierre["fatigue_fmt"],
        # hovertemplate="%{customdata} min/km<extra></extra>",
    ),
    secondary_y=True,
)

fig.update_layout(
    title="Elevation & Predicted Pace Profile",
    xaxis_title="Cumulative Distance (km)",
    hovermode="x unified",
    template="plotly_white",
    legend=dict(orientation="h", y=-0.15),
)
fig.update_yaxes(title_text="Elevation (m)",    secondary_y=False)
fig.update_yaxes(title_text="Pace (min/km)",    secondary_y=True,  autorange="reversed")

fig.show()
# fig.write_html("data/chart_elevation_pace.html", include_plotlyjs="cdn", full_html=False)
# print("✅ Embeddable div saved → data/chart_elevation_pace.html")

# Fit to total time

In [7]:
from trail_pacer.utils import Conversions as conv
conv.chrono_to_min("3:36:00")

216.0

In [8]:
# ── Fit to total time ──────────────────────────────────────────────────
model = PaceModel(pace_10k_min_per_km='3:53', up_sens=0.05, down_sens=-0.02, fatigue_rate=0.06)  # note: down_sens negative
model.fit_to_total_time(df_gpx, total_time_min=conv.chrono_to_min("3:36:00"))  # pass df_gpx, not predicted_splits

# Re-predict AFTER the fit so the chart reflects updated parameters
splits = model.predict_split_times(df_gpx, start_time=datetime(2025, 6, 1, 8, 0))

Fitted parameters  : {'up_sens': np.float64(0.5836123879175997), 'down_sens': np.float64(5.906417966382196e-10), 'fatigue_rate': np.float64(0.08246788607493492)}
Predicted time     : 3:36:00
Target time        : 3:36:00
Residual           : 0.00 min


In [9]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=splits["distance_km"], y=splits["elevation_m"],
        fill="tozeroy", fillcolor="rgba(100,150,255,0.15)",
        line=dict(color="rgba(100,150,255,0.6)", width=1.5),
        name="Elevation (m)",
        hovertemplate="%{y:.0f} m<extra></extra>",
    ),
    secondary_y=False,
)

splits["pace_fmt"]    = splits["predicted_pace"].apply(conv.min_to_chrono)

fig.add_trace(
    go.Scatter(
        x=splits["distance_km"], y=splits["predicted_pace"],
        line=dict(color="tomato", width=2.5),
        name="Predicted Pace",
        customdata=splits["pace_fmt"],
        hovertemplate="%{customdata} min/km<extra></extra>",
    ),
    secondary_y=True,
)
fig.add_trace(
    go.Scatter(
        x=df_temps_pierre["cum_dist"]-1, y=df_temps_pierre["pace"],
        line=dict(color="darkorange", width=2, dash="dot"),
        name="Real Pace",
        # customdata=df_temps_pierre["fatigue_fmt"],
        # hovertemplate="%{customdata} min/km<extra></extra>",
    ),
    secondary_y=True,
)

fig.update_layout(
    title="Elevation & Predicted Pace Profile",
    xaxis_title="Cumulative Distance (km)",
    hovermode="x unified",
    template="plotly_white",
    legend=dict(orientation="h", y=-0.15),
)
fig.update_yaxes(title_text="Elevation (m)",    secondary_y=False)
fig.update_yaxes(title_text="Pace (min/km)",    secondary_y=True,  autorange="reversed")

fig.show()
# fig.write_html("data/chart_elevation_pace.html", include_plotlyjs="cdn", full_html=False)
# print("✅ Embeddable div saved → data/chart_elevation_pace.html")

# Fit to actual pace per split

In [10]:
# ── Fit to actual split paces ──────────────────────────────────────────
# Align Pierre's paces onto the 1-km GPX grid first
import numpy as np

pierre_pace_aligned = np.interp(
    df_gpx["distance_km"].values,       # target x: GPX cumulative distances
    df_temps_pierre["cum_dist"].values,  # source x: Pierre's cumulative distances
    df_temps_pierre["pace"].values       # source y: Pierre's pace per split
)

model = PaceModel(pace_10k_min_per_km='3:53', down_sens=-0.02)  # keep sign consistent
model.fit_to_split_paces(df_gpx, actual_pace_min_per_km=pierre_pace_aligned)  # pass df_gpx

splits = model.predict_split_times(df_gpx, start_time=datetime(2025, 6, 1, 8, 0))

Fitted parameters : {'up_sens': np.float64(0.40404590288868725), 'down_sens': np.float64(0.0), 'fatigue_rate': np.float64(0.1)}
MAE               : 2.413 min/km
RMSE              : 3.437 min/km


In [11]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=splits["distance_km"], y=splits["elevation_m"],
        fill="tozeroy", fillcolor="rgba(100,150,255,0.15)",
        line=dict(color="rgba(100,150,255,0.6)", width=1.5),
        name="Elevation (m)",
        hovertemplate="%{y:.0f} m<extra></extra>",
    ),
    secondary_y=False,
)

splits["pace_fmt"]    = splits["predicted_pace"].apply(conv.min_to_chrono)

fig.add_trace(
    go.Scatter(
        x=splits["distance_km"], y=splits["predicted_pace"],
        line=dict(color="tomato", width=2.5),
        name="Predicted Pace",
        customdata=splits["pace_fmt"],
        hovertemplate="%{customdata} min/km<extra></extra>",
    ),
    secondary_y=True,
)
fig.add_trace(
    go.Scatter(
        x=df_temps_pierre["cum_dist"]-1, y=df_temps_pierre["pace"],
        line=dict(color="darkorange", width=2, dash="dot"),
        name="Real Pace",
        # customdata=df_temps_pierre["fatigue_fmt"],
        # hovertemplate="%{customdata} min/km<extra></extra>",
    ),
    secondary_y=True,
)

fig.update_layout(
    title="Elevation & Predicted Pace Profile",
    xaxis_title="Cumulative Distance (km)",
    hovermode="x unified",
    template="plotly_white",
    legend=dict(orientation="h", y=-0.15),
)
fig.update_yaxes(title_text="Elevation (m)",    secondary_y=False)
fig.update_yaxes(title_text="Pace (min/km)",    secondary_y=True,  autorange="reversed")

fig.show()
# fig.write_html("data/chart_elevation_pace.html", include_plotlyjs="cdn", full_html=False)
# print("✅ Embeddable div saved → data/chart_elevation_pace.html")

In [12]:
splits

,distance_km,elevation_m,elev_diff_m,split_gain_m,split_loss_m,cum_gain_m,cum_loss_m,grade_pct,predicted_pace,split_time_min,cum_time_min,cum_time_str,arrival_time,cum_predicted_pace,pace_fmt
0,0.000000,223.240000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.883333,0.000000,0.000000,0:00:00,2025-06-01 08:00:00.000000000,3.883333,3:53
1,1.000000,189.790097,-33.449903,0.000000,33.449903,0.000000,33.449903,-3.344990,3.983333,3.983333,3.983333,0:03:59,2025-06-01 08:03:58.999999998,7.866667,3:59
2,2.000000,290.503822,100.713725,100.713725,0.000000,100.713725,33.449903,10.071373,8.152630,8.152630,12.135963,0:12:08,2025-06-01 08:12:08.157807852,16.019297,8:09
3,3.000000,471.747558,181.243736,181.243736,0.000000,281.957462,33.449903,18.124374,11.506412,11.506412,23.642376,0:23:39,2025-06-01 08:23:38.542542744,27.525709,11:30
4,4.000000,563.381648,91.634090,91.634090,0.000000,373.591551,33.449903,9.163409,7.985771,7.985771,31.628147,0:31:38,2025-06-01 08:31:37.688813790,35.511480,7:59
5,5.000000,584.935067,21.553419,21.553419,0.000000,395.144970,33.449903,2.155342,5.254190,5.254190,36.882337,0:36:53,2025-06-01 08:36:52.940237604,40.765671,5:15
6,6.000000,602.136209,17.201142,17.201142,0.000000,412.346112,33.449903,1.720114,5.178338,5.178338,42.060676,0:42:04,2025-06-01 08:42:03.640542954,45.944009,5:11
7,7.000000,698.103644,95.967435,95.967435,0.000000,508.313547,33.449903,9.596743,8.460858,8.460858,50.521534,0:50:31,2025-06-01 08:50:31.292035866,54.404867,8:28
8,8.000000,799.745629,101.641985,101.641985,0.000000,609.955532,33.449903,10.164198,8.790136,8.790136,59.311670,0:59:19,2025-06-01 08:59:18.700201356,63.195003,8:47
9,9.000000,610.455515,-189.290113,0.000000,189.290113,609.955532,222.740017,-18.929011,4.783333,4.783333,64.095003,1:04:06,2025-06-01 09:04:05.700201354,67.978337,4:47


# Same with pace fitting

# Fit to total time

In [13]:
# ── Fit to total time ──────────────────────────────────────────────────
model = PaceModel(pace_10k_min_per_km='3:53', up_sens=0.05, down_sens=-0.02, fatigue_rate=0.06)  # note: down_sens negative
model.fit_to_total_time(
    df_gpx,
    total_time_min=conv.chrono_to_min("3:36:00"),
    fit_base_pace=True,
    fit_fatigue=False,   # recommended: fix one to avoid degeneracy
)

# Re-predict AFTER the fit so the chart reflects updated parameters
splits = model.predict_split_times(df_gpx, start_time=datetime(2025, 6, 1, 8, 0))

Fitted parameters  : {'base_pace': np.float64(6.14598065272146), 'up_sens': np.float64(0.24335966460113728), 'down_sens': np.float64(0.0)}
Predicted time     : 3:36:00
Target time        : 3:36:00
Residual           : 0.00 min


In [14]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=splits["distance_km"], y=splits["elevation_m"],
        fill="tozeroy", fillcolor="rgba(100,150,255,0.15)",
        line=dict(color="rgba(100,150,255,0.6)", width=1.5),
        name="Elevation (m)",
        hovertemplate="%{y:.0f} m<extra></extra>",
    ),
    secondary_y=False,
)

splits["pace_fmt"]    = splits["predicted_pace"].apply(conv.min_to_chrono)

fig.add_trace(
    go.Scatter(
        x=splits["distance_km"], y=splits["predicted_pace"],
        line=dict(color="tomato", width=2.5),
        name="Predicted Pace",
        customdata=splits["pace_fmt"],
        hovertemplate="%{customdata} min/km<extra></extra>",
    ),
    secondary_y=True,
)
fig.add_trace(
    go.Scatter(
        x=df_temps_pierre["cum_dist"]-1, y=df_temps_pierre["pace"],
        line=dict(color="darkorange", width=2, dash="dot"),
        name="Real Pace",
        # customdata=df_temps_pierre["fatigue_fmt"],
        # hovertemplate="%{customdata} min/km<extra></extra>",
    ),
    secondary_y=True,
)

fig.update_layout(
    title="Elevation & Predicted Pace Profile",
    xaxis_title="Cumulative Distance (km)",
    hovermode="x unified",
    template="plotly_white",
    legend=dict(orientation="h", y=-0.15),
)
fig.update_yaxes(title_text="Elevation (m)",    secondary_y=False)
fig.update_yaxes(title_text="Pace (min/km)",    secondary_y=True,  autorange="reversed")

fig.show()
# fig.write_html("data/chart_elevation_pace.html", include_plotlyjs="cdn", full_html=False)
# print("✅ Embeddable div saved → data/chart_elevation_pace.html")

# Fit to actual pace per split

In [15]:
# ── Fit to actual split paces ──────────────────────────────────────────
# Align Pierre's paces onto the 1-km GPX grid first
import numpy as np

pierre_pace_aligned = np.interp(
    df_gpx["distance_km"].values,       # target x: GPX cumulative distances
    df_temps_pierre["cum_dist"].values,  # source x: Pierre's cumulative distances
    df_temps_pierre["pace"].values       # source y: Pierre's pace per split
)

model = PaceModel(pace_10k_min_per_km='3:53')
model.fit_to_split_paces(
    df_gpx,
    actual_pace_min_per_km=pierre_pace_aligned,
    fit_base_pace=True,   # all four params free, data is rich enough
)

splits = model.predict_split_times(df_gpx, start_time=datetime(2025, 6, 1, 8, 0))

Fitted parameters : {'base_pace': np.float64(6.842630407452538), 'up_sens': np.float64(0.25728091207798404), 'down_sens': np.float64(0.0), 'fatigue_rate': np.float64(0.03467650558662372)}
MAE               : 2.336 min/km
RMSE              : 2.991 min/km


In [16]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=splits["distance_km"], y=splits["elevation_m"],
        fill="tozeroy", fillcolor="rgba(100,150,255,0.15)",
        line=dict(color="rgba(100,150,255,0.6)", width=1.5),
        name="Elevation (m)",
        hovertemplate="%{y:.0f} m<extra></extra>",
    ),
    secondary_y=False,
)

splits["pace_fmt"]    = splits["predicted_pace"].apply(conv.min_to_chrono)

fig.add_trace(
    go.Scatter(
        x=splits["distance_km"], y=splits["predicted_pace"],
        line=dict(color="tomato", width=2.5),
        name="Predicted Pace",
        customdata=splits["pace_fmt"],
        hovertemplate="%{customdata} min/km<extra></extra>",
    ),
    secondary_y=True,
)
fig.add_trace(
    go.Scatter(
        x=df_temps_pierre["cum_dist"]-1, y=df_temps_pierre["pace"],
        line=dict(color="darkorange", width=2, dash="dot"),
        name="Real Pace",
        # customdata=df_temps_pierre["fatigue_fmt"],
        # hovertemplate="%{customdata} min/km<extra></extra>",
    ),
    secondary_y=True,
)

fig.update_layout(
    title="Elevation & Predicted Pace Profile",
    xaxis_title="Cumulative Distance (km)",
    hovermode="x unified",
    template="plotly_white",
    legend=dict(orientation="h", y=-0.15),
)
fig.update_yaxes(title_text="Elevation (m)",    secondary_y=False)
fig.update_yaxes(title_text="Pace (min/km)",    secondary_y=True,  autorange="reversed")

fig.show()
# fig.write_html("data/chart_elevation_pace.html", include_plotlyjs="cdn", full_html=False)
# print("✅ Embeddable div saved → data/chart_elevation_pace.html")